# K-Means: sensitivity to the number of clusters (`k`)

## 1. Imports and setup

In [ ]:
import sys, os, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

def _find_root() -> Path:
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / 'src').is_dir() and (c / 'data').is_dir():
            return c
    return here

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f'Project root: {ROOT}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity

from src.preprocess import load_data, get_audio_features, AUDIO_FEATURES
from src.recommender import (
    interpret_playlist, score_songs_by_audio,
    AUDIO_FEATURES as REC_AUDIO_FEATURES,
)
from src.embeddings import create_playlist_embedding
from src.evaluate import precision_at_k
from src.test_set import TEST_PLAYLISTS, HELD_OUT_PLAYLISTS, get_ground_truth_indices

OUT_DIR = ROOT / 'data' / 'cluster_comparison_results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SILHOUETTE_SAMPLE = 50_000
TOP_K = 10
ALPHA = 0.9
K_VALUES = [5, 10, 20, 40, 60, 80, 100]
print('Imports done.')

## 2. Load dataset and prepare audio matrix

In [ ]:
df = load_data('data/spotify_data.csv')
audio_matrix, scaler = get_audio_features(df)
text_embeddings = np.load('data/text_embeddings.npy')

rng = np.random.default_rng(RANDOM_STATE)
sil_idx = rng.choice(len(audio_matrix), size=SILHOUETTE_SAMPLE, replace=False)
sil_X = audio_matrix[sil_idx]

print(f'Songs:               {len(df):,}')
print(f'Audio matrix shape:  {audio_matrix.shape}')
print(f'Silhouette sample:   {SILHOUETTE_SAMPLE:,}')

## 3. Helper functions

In [ ]:
def compute_centroids(X, labels):
    return {int(c): X[labels == c].mean(axis=0)
            for c in np.unique(labels) if c >= 0}

def get_relevant_clusters_generic(target_features, scaler, centroids, top_n=3):
    feature_vec = np.array([[target_features[f] for f in REC_AUDIO_FEATURES]])
    normalized = scaler.transform(feature_vec)[0]
    cluster_ids = list(centroids.keys())
    if not cluster_ids:
        return set()
    cmat = np.array([centroids[c] for c in cluster_ids])
    d = np.linalg.norm(cmat - normalized, axis=1)
    return set(int(cluster_ids[i]) for i in d.argsort()[:top_n])

def evaluate_recommender(df_with_clusters, scaler, centroids, playlists):
    results = []
    for pl in playlists:
        interp = interpret_playlist(pl['name'])
        relevant = get_relevant_clusters_generic(interp['features'], scaler, centroids)
        year_range = interp.get('year_range')
        audio_scores = score_songs_by_audio(
            df_with_clusters, interp['features'], interp['genres'],
            relevant if relevant else None, year_range,
        )
        name_text_vec = create_playlist_embedding(pl['name'])
        text_sim = cosine_similarity(name_text_vec, text_embeddings)[0]
        text_sim_norm = (text_sim - text_sim.min()) / (text_sim.max() - text_sim.min() + 1e-8)
        final = ALPHA * audio_scores + (1 - ALPHA) * text_sim_norm
        top_idx = final.argsort()[-TOP_K:][::-1].tolist()
        truth = get_ground_truth_indices(df_with_clusters, pl)
        results.append({'playlist': pl['name'],
                        'precision@10': precision_at_k(top_idx, list(truth), TOP_K)})
    return results

def intrinsic_metrics(labels):
    sub = labels[sil_idx]
    if len(set(sub.tolist())) < 2:
        return np.nan, np.nan, np.nan
    return (silhouette_score(sil_X, sub),
            davies_bouldin_score(sil_X, sub),
            calinski_harabasz_score(sil_X, sub))

print('Helpers ready.')

## 4. Run the sweep

In [ ]:
sweep_results = []

for k in K_VALUES:
    print(f'\n=== k = {k} ===')
    t0 = time.time()
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=3)
    labels = km.fit_predict(audio_matrix)
    fit_runtime = time.time() - t0

    sil, db, ch = intrinsic_metrics(labels)
    df_local = df.copy()
    df_local['cluster'] = labels
    centroids = compute_centroids(audio_matrix, labels)

    dev = evaluate_recommender(df_local, scaler, centroids, TEST_PLAYLISTS)
    held = evaluate_recommender(df_local, scaler, centroids, HELD_OUT_PLAYLISTS)
    dev_mean = float(np.mean([r['precision@10'] for r in dev]))
    held_mean = float(np.mean([r['precision@10'] for r in held]))
    total_runtime = time.time() - t0

    print(f'  fit: {fit_runtime:.0f}s  total: {total_runtime:.0f}s')
    print(f'  Silhouette: {sil:.4f}   Davies-Bouldin: {db:.4f}   Calinski-Harabasz: {ch:.0f}')
    print(f'  Dev mean Prec@10:      {dev_mean:.4f}')
    print(f'  Held-out mean Prec@10: {held_mean:.4f}')

    sweep_results.append({
        'k': k,
        'fit_runtime_sec': round(fit_runtime, 1),
        'silhouette': round(sil, 4),
        'davies_bouldin': round(db, 4),
        'calinski_harabasz': round(ch, 0),
        'dev_mean_p@10': round(dev_mean, 4),
        'held_out_mean_p@10': round(held_mean, 4),
    })

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(OUT_DIR / 'k_sweep.csv', index=False)
sweep_df

## 5. Plot the sweep

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))

panels = [
    ('silhouette',        'Silhouette  (higher better)',        '#1f77b4'),
    ('davies_bouldin',    'Davies-Bouldin  (lower better)',     '#d62728'),
    ('calinski_harabasz', 'Calinski-Harabasz  (higher better)', '#2ca02c'),
]
for ax, (col, title, color) in zip(axes[:3], panels):
    ax.plot(sweep_df['k'], sweep_df[col], 'o-', color=color, linewidth=2, markersize=8)
    mark = sweep_df[sweep_df['k'] == 20]
    if len(mark):
        ax.plot(mark['k'], mark[col], 's', color='red',
                markersize=14, markerfacecolor='none', markeredgewidth=2,
                label='k=20 (chosen)')
        ax.legend()
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('number of clusters (k)')
    ax.grid(alpha=0.3)

ax = axes[3]
ax.plot(sweep_df['k'], sweep_df['dev_mean_p@10'], 'o-', color='#1f77b4',
        linewidth=2, markersize=8, label='Dev')
ax.plot(sweep_df['k'], sweep_df['held_out_mean_p@10'], 'o-', color='#ff7f0e',
        linewidth=2, markersize=8, label='Held-out')
mark = sweep_df[sweep_df['k'] == 20]
if len(mark):
    ax.plot(mark['k'], mark['dev_mean_p@10'], 's', color='red',
            markersize=14, markerfacecolor='none', markeredgewidth=2)
    ax.plot(mark['k'], mark['held_out_mean_p@10'], 's', color='red',
            markersize=14, markerfacecolor='none', markeredgewidth=2)
ax.set_title('End-to-end Precision@10  (higher better)', fontsize=11)
ax.set_xlabel('number of clusters (k)')
ax.grid(alpha=0.3)
ax.legend()

plt.suptitle('K-Means: metric sensitivity to k', fontsize=14, y=1.04)
plt.tight_layout()
plt.savefig(OUT_DIR / 'viz_k_sweep.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Verdict

In [ ]:
best_sil_k  = int(sweep_df.loc[sweep_df['silhouette'].idxmax(), 'k'])
best_db_k   = int(sweep_df.loc[sweep_df['davies_bouldin'].idxmin(), 'k'])
best_ch_k   = int(sweep_df.loc[sweep_df['calinski_harabasz'].idxmax(), 'k'])
best_dev_k  = int(sweep_df.loc[sweep_df['dev_mean_p@10'].idxmax(), 'k'])
best_held_k = int(sweep_df.loc[sweep_df['held_out_mean_p@10'].idxmax(), 'k'])

row20 = sweep_df[sweep_df['k'] == 20].iloc[0]

print('Best k by metric:')
print(f'  Silhouette:        k={best_sil_k}')
print(f'  Davies-Bouldin:    k={best_db_k}')
print(f'  Calinski-Harabasz: k={best_ch_k}')
print(f'  Dev Prec@10:       k={best_dev_k}')
print(f'  Held-out Prec@10:  k={best_held_k}')
print()
print('k = 20 results:')
print(f'  Silhouette: {row20["silhouette"]:.4f}   '
      f'Davies-Bouldin: {row20["davies_bouldin"]:.4f}   '
      f'Calinski-Harabasz: {row20["calinski_harabasz"]:.0f}')
print(f'  Dev Prec@10:      {row20["dev_mean_p@10"]:.4f}')
print(f'  Held-out Prec@10: {row20["held_out_mean_p@10"]:.4f}')